# Lab 08 · Human-in-the-loop and agent memory

**Prepared by Mustafa Assaf** · Lecture 3: Debate, trace, decide · ~15–20 min core work (estimate; setup and extras take longer)

## Human feedback and memory: when is a retry safe?
**Start with a problem:** A human’s retry advice for a safe read could be dangerous if reused for a payment charge just because the cases sound similar.

**In this lab:** Store the approved advice with its scope, then check applicability before reuse.

**So far:** the checkout exercise introduced resilience patterns. **New:** a human review loop and a memory of accepted advice. We revisit the example's distinction between an idempotent read and a non-idempotent payment charge.

The initial proposal is conservative: do not retry a timed-out Pricing lookup. A scripted reviewer stands in for a human and approves a bounded retry because this lookup is idempotent and the failure is transient. No person is asked to approve in this lab. The accepted lesson carries an explicit `idempotent=True`, `transient=True` scope. A similar-looking Payments case must not receive it.

Short-term notes carry the feedback through this review. A Chroma archive carries the accepted lesson across decisions. This demo uses in-memory storage, so restarting Python clears it; production persistence is a separate choice. Practice answers are scripted, and the two held-out cases demonstrate measurement rather than prove learning.

**At a glance**

![Lab 08 at a glance: Council proposal (retry or do not retry), then Human review (verdict + reason), then Council revises (reads short-term notes), then Long-term memory (lesson + explicit scope), then Recall (similar AND applicable), then Held-out score (before vs after). Then back to Human review: at most 2 reviews.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/08.svg)

## Think about the example

Pricing and Catalog perform read-only lookups, while Payments performs a card charge. A timed-out charge may already have succeeded, so repeating it blindly can charge twice.

**Concepts:** Idempotent reads versus non-idempotent charges; Bounded retries and deduplicated effects.

Use the scenario above to consider these questions before running the code:

1. Why must a timeout not automatically trigger another payment charge?
2. Why does semantic similarity alone not establish that a stored lesson applies?

## The idea in context

Human review can correct a proposal and supply a scoped lesson. This lab uses a scripted reviewer, and retrieval checks both relevance and applicability.

Similarity is not applicability. Check known scope conditions before inserting a lesson into a prompt.

| Key term | Meaning |
| --- | --- |
| **Human in the loop** | A design in which a person approves or corrects the system's decisions. |
| **Short-term memory** | Working notes for the current task, passed in the prompt. |
| **Long-term memory** | Lessons stored across tasks and recalled when a similar case appears. |
| **Held-out case** | A case not used for feedback, reserved to test whether the approach generalizes. |

## Pattern · Human in the Loop

*Multi-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide.*

| | |
| --- | --- |
| **Context** | Agents can propose actions whose consequences are hard to reverse. |
| **Problem** | Fully autonomous execution treats routine and high-risk decisions alike, and leaves no accountable decision point. |
| **Solution** | Classify each step by risk, run the low-risk ones, and gate high-risk ones behind explicit human approval, with an audit trail. |
| **Use when** | Decisions affect people or money; reversibility varies; auditability is required. |
| **Trade-off** | Safer and accountable, but approvals interrupt the flow and can become a bottleneck. |
| **Flow** | Plan → Classify risk → Approve? → Execute + audit |
| **Quality attributes** | Benefits: Safety, Reliability, Auditability, Usability. Liabilities: Latency, Cost, Scalability, Usability, Reliability. |
| **In this lab** | A scripted reviewer stands in for the human architect and returns approval or rejection with reasons; the notebook does not request real human input. Self-study Lab 10 adds the risk classification: only contested cases go to the council. |

## Pattern · Scratchpad Memory

*Memory pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide.*

| | |
| --- | --- |
| **Context** | Multi-step work produces partial results and pending questions that are not long-term facts. |
| **Problem** | Keeping work in progress only in the message history makes it noisy, fragile, and hard to update cleanly. |
| **Solution** | Give the agent a dedicated block of working notes that it appends to, replaces, or clears as the work evolves. |
| **Use when** | Tasks span several turns; intermediate state matters; working notes should stay separate from durable facts. |
| **Trade-off** | Makes state explicit, but unmanaged notes consume context and can preserve obsolete assumptions. |
| **Flow** | Work → Scratchpad → Update notes → Resume |
| **Quality attributes** | Benefits: Reliability, Observability, Maintainability. Liabilities: Cost, Reliability, Privacy. |
| **In this lab** | `short_term` holds the human's feedback while one decision is made, and every revision prompt includes it. Once the decision is archived, the scratchpad is cleared, so an obsolete note cannot leak into the next case. |

## Pattern · Archival Memory

*Memory pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide.*

| | |
| --- | --- |
| **Context** | Long-lived agents accumulate more knowledge than can stay in the context window. |
| **Problem** | Keeping everything in context is expensive and eventually impossible, while dropping old facts loses useful knowledge. |
| **Solution** | Move durable facts into archival storage, and search it semantically so that only the relevant facts return to the context. |
| **Use when** | Facts accumulate over time; semantic retrieval is acceptable; long-term persistence matters. |
| **Trade-off** | Scales far beyond the context, but retrieval can miss relevant facts or return misleading neighbours. |
| **Flow** | Fact → Archive → Semantic search → Recall into context |
| **Quality attributes** | Benefits: Scalability, Cost, Data freshness. Liabilities: Latency, Reliability, Maintainability. |
| **In this lab** | The accepted decision and its reasons are archived in a Chroma collection and recalled by semantic search. Step 4 shows the pattern's liability: a similar timeout may hide a non-idempotent effect. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit', 'chromadb']):
    %pip install -q "agentic-ai-coursekit[rag] @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · A proposal and a stand-in council

This one-call proposer stands in for a complete council. The policy is a simplified teaching policy: never retry a non-idempotent action automatically; absent approved advice, choose the conservative option. Its output is a design recommendation, not an executed network request.

In [ ]:
case = "ShopEasy Pricing: an idempotent quote lookup timed out during a transient network fault."
proposal = {"option": "do not retry", "reason": "A timeout leaves an uncertain outcome."}
OPTIONS = ["retry", "do not retry"]
POLICY = "Retry only transient idempotent effects with approved advice; otherwise do not retry."

def council(case, notes="", example=""):
    """One proposal, using policy and applicable human-approved advice."""
    prompt = (f"Policy: {POLICY}\n{notes}\nCase: {case}\n"
              f"Reply with exactly one label from {OPTIONS}. Retry means bounded backoff with jitter.")
    option = ask(prompt, example=example).strip().rstrip(".").lower()
    if option not in OPTIONS:
        raise ValueError(f"Unknown option: {option!r}")
    return option

print("Proposed:", proposal)

## Step 2 · Simulate human review and define its scope

The `human_review` function returns scripted feedback in both practice and live modes; it does not ask a person. The reviewer's reason explains the revision. The accepted lesson is restricted to transient failures of idempotent operations; approval for one safe read is not permission to repeat a charge. At most two reviews keep the loop bounded.

In [ ]:
def human_review(option):
    if option == "do not retry":
        return {"verdict": "reject", "reason": "This read is idempotent and the fault is transient; use bounded backoff and jitter."}
    return {"verdict": "accept", "reason": "A bounded retry is safe for this transient failed read.", "idempotent": True, "transient": True}

option, short_term = proposal["option"], []
for n in (1, 2):
    review = human_review(option)
    print(f"Review {n}: {option} -> {review['verdict']}: {review['reason']}")
    if review["verdict"] == "accept":
        break
    short_term.append(review["reason"])
    option = council(case, "Human feedback on this case: " + "\n".join(short_term), example="retry")
assert review["verdict"] == "accept", "No accepted proposal; the human must decide directly."

## Step 3 · Archive the lesson with an applicability condition

The vector search finds candidates by meaning. The metadata filter checks the human-approved condition **before** a recalled lesson enters the prompt. It uses an input fact, not an expected test answer. The explicit `candidates_only=True` view shows similarity alone and is for inspection, not advice. Applying the lesson requires known idempotency and transient-failure facts.

The database stores the lesson alongside the human-approved `idempotent` and `transient` scope. `recall` returns pairs of distance and lesson text; a lower distance means a closer match.

In [ ]:
import chromadb
from chromadb.config import Settings
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

db = chromadb.Client(Settings(anonymized_telemetry=False))
memory = db.get_or_create_collection("retry_lessons", embedding_function=DefaultEmbeddingFunction(),
                                    configuration={"hnsw": {"space": "cosine"}})
lesson = "\n".join([f"Past case: {case}", *short_term, f"Accepted: {option}. {review['reason']}"])
memory.upsert(ids=["D1"], documents=[case], metadatas=[{"lesson": lesson, "idempotent": review["idempotent"], "transient": review["transient"]}])
short_term.clear()
print("Working notes:", len(short_term), "| archived lessons:", memory.count())

def recall(new_case, idempotent=None, transient=None, max_distance=0.6, *, candidates_only=False):
    if not candidates_only and (type(idempotent) is not bool or type(transient) is not bool):
        return []  # unknown scope cannot receive advice
    where = None if candidates_only else {"$and": [{"idempotent": idempotent}, {"transient": transient}]}
    found = memory.query(query_texts=[new_case], n_results=1, where=where)
    results = []
    for distance, record in zip(found["distances"][0], found["metadatas"][0]):
        if distance <= max_distance:
            results.append((round(distance, 2), record["lesson"]))
    return results

print(recall(case, idempotent=True, transient=True)[0][1])

## Step 4 · Hold out two new cases

Neither case gave feedback. Both can resemble the archived network failure. Their effect differs: repeating a Catalog read is idempotent, repeating the Payments charge is not. `idempotent` is provided by the service contract; the model must not invent it.

In [ ]:
held_out = [
    ("ShopEasy Catalog: an idempotent product lookup timed out during a transient network fault.", True, True, "retry"),
    ("ShopEasy Payments: a non-idempotent card charge timed out; the charge outcome is uncertain.", False, True, "do not retry"),
]
for c, safe, transient, _ in held_out:
    print("Semantic candidates:", len(recall(c, candidates_only=True)), "| applicable:", len(recall(c, idempotent=safe, transient=transient)), "|", c)
assert not recall(held_out[1][0], idempotent=False, transient=True), "A read-only retry lesson must not reach the payment case."
assert not recall(held_out[0][0], idempotent=None, transient=True), "Unknown scope needs review."
assert not recall(held_out[0][0], idempotent=True, transient=False), "Permanent failures need different advice."

## Step 5 · Measure before and after, without requiring a win

Only applicable lessons enter the after-memory prompt. Report flat or worse results honestly. A two-case score is a demonstration; a project needs a larger held-out benchmark and repeated live runs.

Run each held-out case once without memory and once with applicable memory. The printed score is an observation; the lesson may help, hurt, or make no difference.

In [ ]:
assert case not in [c for c, _, _, _ in held_out], "Feedback cases cannot be test cases."
scripted = {"before": ["do not retry", "do not retry"], "after": ["retry", "do not retry"]}
scores = {}
for run in ("before", "after"):
    answers = []
    for (c, safe, transient, expected), example in zip(held_out, scripted[run]):
        lessons = []
        if run == "after":
            for distance, text in recall(c, idempotent=safe, transient=transient):
                lessons.append(text)
        notes = "Applicable approved lessons:\n" + "\n".join(lessons) if lessons else ""
        answers.append(council(c, notes, example=example))
    scores[run] = sum(a == expected for a, (_, _, _, expected) in zip(answers, held_out))
    print(run, answers, f"{scores[run]}/{len(held_out)} match the human")
print("Score change:", scores["after"] - scores["before"])

## What you should see

The scripted reviewer rejects `do not retry` for the idempotent Pricing lookup, then accepts `retry`. Short-term notes are cleared after archiving. Catalog can recall the lesson; Payments and cases with an unknown or permanent failure receive none after filtering. Scripted scores change from 1/2 to 2/2. Live scores may differ.

## Try it

1. Remove the metadata filter. Why could a similar timeout now pass unsafe advice to Payments?
2. Try `recall(case, idempotent=None, transient=True)`. Why does an unknown idempotency contract return no applicable lesson? Which service owner could establish it?
3. Lower `max_distance` so no lesson is recalled. Does the score still improve?
4. An outbox relay may publish the same event twice after a retry. Why must a consumer deduplicate by event ID and commit that ID with the business effect?

## Takeaway

A memory needs applicability as well as similarity. Human feedback approved a bounded retry for a transient failure of one class of safe effect, not for every timeout. The agentic lesson is review, explicit state, archival retrieval, and held-out evaluation; the microservices lesson is idempotency and safe retry.